In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tables_io

In [ ]:
tf = tables_io.read('output_unrec_bl_flagship.pq')
tc = tables_io.read('output_unrec_bl_cardinal.pq')

In [ ]:
i_mask_tc = tc['mag_i_lsst'] < 25.5
i_mask_tf = tf['mag_i_lsst'] < 25.5
_ = plt.hist(tc['n_obj'][i_mask_tc][0:1000000], bins=np.linspace(0.5, 8.5, 8), label='cardinal', alpha=0.2)
_ = plt.hist(tf['n_obj'][i_mask_tf][0:1000000], bins=np.linspace(0.5, 8.5, 8), label='flagship', alpha=0.2)
_ = plt.yscale('log')
_ = plt.xlabel('Number of objects in blend')
_ = plt.ylabel('Counts / 1M objects')
_ = plt.legend()
plt.gcf().savefig('n_objects_in_blend.png')

In [ ]:
tc_mask = np.bitwise_and(i_mask_tc, tc['n_obj'] > 1)
tf_mask = np.bitwise_and(i_mask_tf, tf['n_obj'] > 1)

plt.hist(tc['redshift'][tc_mask]/tc['z_weighted'][tc_mask], bins=np.logspace(-1, 1, 101), alpha=0.2, label='Cardinal')
tf_mask = tf['n_obj'] > 1
plt.hist(tf['redshift'][tf_mask]/tf['z_weighted'][tf_mask], bins=np.logspace(-1, 1, 101), alpha=0.2, label='Flagship')
_ = plt.xscale('log')
_ = plt.yscale('log')
_ = plt.ylabel('Number of objects [1M object / 0.02 dex]')
_ = plt.xlabel(r'$z_{\rm brightest}/z_{\rm weighted}$')
_ = plt.legend()
_ = plt.gcf().savefig('redshift_ratio.png')

In [ ]:
_ = plt.hist(
    1-(tc['brightest_flux'][tc_mask]/tc['total_flux'][tc_mask])[0:100000],
    bins=np.linspace(0, 1, 101), alpha=0.2, label='Cardinal'
)
_ = plt.hist(
    1-(tf['brightest_flux'][tf_mask]/tf['total_flux'][tf_mask])[0:100000],
    bins=np.linspace(0, 1, 101), alpha=0.2, label='Flagship'
)
_ = plt.ylabel('Number of objects [100K object / 0.01]')
_ = plt.xlabel(r'Contamination: $1 - F_{i, \rm brightest}/F_{i, \rm total}$')

_ = plt.legend()
plt.gcf().savefig('flux_contamination.png')

In [ ]:
zbins = np.linspace(0, 3, 61)
bin_centers =  0.5*(zbins[1:] + zbins[0:-1])
h2_c = np.histogram2d(tc['redshift'], tc['n_obj'] > 1, bins=(zbins, np.linspace(-0.5, 1.5, 3)))[0]
h2_f = np.histogram2d(tf['redshift'], tf['n_obj'] > 1, bins=(zbins, np.linspace(-0.5, 1.5, 3)))[0]


In [ ]:
frac_c = h2_c[:,1] / h2_c.sum(axis=1)
frac_c_err = np.sqrt((frac_c*(1-frac_c)*h2_c.sum(axis=1)))/h2_c.sum(axis=1)
frac_f = h2_f[:,1] / h2_f.sum(axis=1)
frac_f_err = np.sqrt((frac_f*(1-frac_f)*h2_f.sum(axis=1)))/h2_f.sum(axis=1)

In [ ]:
_ = plt.errorbar(bin_centers, frac_c, yerr=frac_c_err, fmt='o', label='cardinal')
_ = plt.errorbar(bin_centers, frac_f, yerr=frac_f_err, fmt='o', label='flagship')

_ = plt.xlabel('z')
_ = plt.ylabel('Unrecognized blend fraction')
_ = plt.ylim(0., 0.15)
_ = plt.legend()
plt.gcf().savefig('blend_fractions.png')